# Workshop: Data Wrangler

**Pedagogical goal**: use the visual interface **Data Wrangler** in a Fabric notebook to explore, clean and transform data, then retrieve the Python code generated automatically.

**Prerequisite**: Lakehouse`dwg_lakehouse`attached to this notebook, file`consumption_raw.csv`loaded in **Files** (see Part 1).

> The cells marked ** The "generated" code cells correspond to what Data Wrangler produces via **Add code to notebook**: you can either run them directly or get them yourself from the interface and then compare them.

## Background and data

The source file is **`consumption_raw.csv`**: gross electricity consumption with voluntary quality problems.

| Problem | Exemple |
| --- | --- |
| Formats de dates mixtes | `2025-01-03T04:30:00`, `17/01/2025 18:15:00`, `2025-01-22 01:00:00` |
| Missing value | `consumption_mw`empty (`status = ERROR`) |
| Valeurs aberrantes | `consumption_mw = -888 / -777 / -999` (`status = ERROR`) |
| Coupons | strictly identical lines (retransmissions) |
| Unable negative | `voltage_v` / `frequency_hz`negative on lines`OK` |

## Part 1 — Preparing the environment

1. Ouvrir https://app.fabric.microsoft.comthen your workspace
2. Create a **Lakehouse**`dwg_lakehouse`(or use an existing one)
3. In the Lakehouse, open **Files** → **Upload** →`consumption_raw.csv`
4. Attach this Lakehouse to this notebook (sign **Explorer**)

### 1.2 — Load data

In [ ]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv("Files/consumption_raw.csv")
    .toPandas()
)

display(df)

**To be verified**: columns`timestamp`, `site_id`, `consumption_mw`, `voltage_v`, `frequency_hz`, `status`; heterogeneous dates ; a line with`consumption_mw`empty and`status = ERROR`.

## Part 2 — Open Data Wrangler

1. Click on the cell containing`df`
2. Notebook toolbar: **Data** → **Launch Data Wrangler**
3. Choose DataFrame **`df`**

Left panel = operations; centre = preview; right = statistical summary by column. Columns with missing values are reported.

## Part 3 — Exploring the data

- `timestamp` : type `object`(string), not yet gone
- `consumption_mw`: 1 missing value visible, and an outlier value`-888`
- `status` : 2 valeurs distinctes (`OK`, `ERROR`)
- These abnormal lines all correspond to`status = ERROR`

Python Verification:

In [ ]:
print(df.shape)
print(df['status'].value_counts())
print("consumption_mw vide :", (df['consumption_mw'].isna()).sum())
print("consumption_mw < 0  :", (df['consumption_mw'].astype(float) < 0).sum())

## Part 4 — Clean up with Data Wrangler

**4.1 Delete ERROR lines** : **Filter** → *Filter rows* → Column`status`, Condition `Not equal to`, Value `ERROR` → **Apply**.

**4.2 Convertir `consumption_mw`** : **Change column type** → `consumption_mw` → `Decimal (float64)` → **Apply**.

**4.3 Standardiser `timestamp`** —  Use **Custom operation**:

```python
import pandas as pd
df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed', dayfirst=True)
```

**4.4 (optionnel) Renommer** `consumption_mw` en `consumption_MW`with **Rename column**.

## Part 5 — Code generated (Add code to notebook)

Click **Add code to notebook**: Data Wrangler inserts a cell like this.

In [ ]:
# Code generated by Data Wrangler for pandas DataFrame
import pandas as pd

def clean_data(df):
    # Filter rows based on column: 'status'
    df = df[df['status'] != "ERROR"]
    # Change column type to float64 for column: 'consumption_mw'
    df = df.astype({'consumption_mw': 'float64'})
    df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed', dayfirst=True)
    # Rename column 'consumption_mw' to 'consumption_MW'
    df = df.rename(columns={'consumption_mw': 'consumption_MW'})
    return df

df_clean = clean_data(df.copy())
display(df_clean)

> The code is **reusable and versionable** (Git, pipeline).

## Part 6 — Saving in Lakehouse

In [ ]:
spark_df = spark.createDataFrame(df_clean)

spark_df.write.mode("overwrite").format("delta").saveAsTable("bronze_consumption")

Check in Lakehouse that the table **`bronze_consumption`** appears under **Tables**.

---
## Part 7 — Deducing rows (Drop duplicate rows)

**Scenario**: measures were broadcast; file contains strictly identical lines as filter`ERROR` ne supprime pas.

### 7.1 — Measuring duplicates

In [ ]:
print("Lignes :", len(df_clean))
print("Doublons exacts :", df_clean.duplicated().sum())

**Intended**:`Lignes : 17271`and`Doublons exacts : 821`.

### 7.2 — To deduce
**Launch Data Wrangler**`df_clean`→ **Drop duplicate rows** (all columns) → **Apply** → **Add code to notebook**. Rename function and output not to overwrite`df_clean` :

In [ ]:
def dedup_data(df_clean):
    # Drop duplicate rows across all columns
    df_clean = df_clean.drop_duplicates()
    return df_clean

df_dedup = dedup_data(df_clean.copy())
display(df_dedup)

### 7.3 — Check

Expected:`16450`, `0`, `0`.

In [ ]:
print("Lignes :", len(df_dedup))
print("Doublons restants :", df_dedup.duplicated().sum())
print("Doublons sur (timestamp, site_id) :", df_dedup.duplicated(subset=["timestamp", "site_id"]).sum())

> 💡 `drop_duplicates(subset=["timestamp", "site_id"], keep="last")`deduces on a **trade key**.

---
## Part 8 — Replace outliers with nulls (Find and replace)

**Scenario**:`voltage_v`and`frequency_hz`contain possible **negative** on lines`OK` :

| Cas | Exemple | Colonnes |
| --- | --- | --- |
| Error code | `-999.0` | `voltage_v`,`frequency_hz` |
| Sign error | `-228.4` | `voltage_v` |

The invalid measure is replaced by a null value (the line remains).

### 8.1 — Identify negatives

In [ ]:
print("voltage_v < 0   :", (df_dedup["voltage_v"].astype(float) < 0).sum())
print("frequency_hz < 0:", (df_dedup["frequency_hz"].astype(float) < 0).sum())

**Intended**:`14`and`5` (dont 8 `-999.0`on`voltage_v`and 5 on`frequency_hz`).

### 8.2 — Convert to digital
**Launch Data Wrangler**`df_dedup` → **Change column type** → `voltage_v`then`frequency_hz` → `Decimal (float64)`. (The minimum`voltage_v`then appears at -999.)

### 8.3 — Find and replace  🖱️
**Find and replace** : Target column `voltage_v`, Old value `-999`, New value vide / `NaN`→ **Apply** ; Idem for`frequency_hz`.

> If your version does not accept an empty value as a replacement, use 8.4.

### 8.4 - All negatives (Custom operation)
Sign errors cannot be listed one by one:

In [ ]:
import numpy as np

def replace_negatives(df_dedup):
    for col in ['voltage_v', 'frequency_hz']:
        s = df_dedup[col].astype('float64')
        df_dedup[col] = s.where(s >= 0, np.nan)
    return df_dedup

df_nonneg = replace_negatives(df_dedup.copy())
print(df_nonneg[["voltage_v", "frequency_hz"]].isna().sum())
print("min voltage_v :", df_nonneg["voltage_v"].min())

**Intended**: 14`NaN`in`voltage_v`, 5 in`frequency_hz`, minimum de `voltage_v`realistic (~190 V).

> 💡 `NaN`rather than`0`(false averages) or line deletion (lose a valid consumption).

---
## Part 9 — Cut and merge text (Split text / Merge)

`site_id` (`SITE_COM_001`) = prefix + **type** (`COM`, `IND`, `RES`) + **number**.

### 9.1 — Split text  🖱️
**Launch Data Wrangler**`df_nonneg` → **Split text** (Target column `site_id`, Delimiter `_`) → **Apply** ; rename columns obtained in`site_type`and`site_num`(**Rename column**), delete column`SITE` (**Drop column**).

### 9.2 — Merge (Custom operation)  🖱️

Equivalent code (column names generated by Split text may vary):

In [ ]:
def split_merge_site(df_nonneg):
    # Split text on column: 'site_id'
    parts = df_nonneg['site_id'].str.split('_', expand=True)
    df_nonneg['site_type'] = parts[1]
    df_nonneg['site_num'] = parts[2]
    # Merge : libellé lisible
    type_labels = {'COM': 'Commercial', 'IND': 'Industriel', 'RES': 'Residentiel'}
    df_nonneg['site_label'] = df_nonneg['site_type'].map(type_labels) + ' #' + df_nonneg['site_num']
    return df_nonneg

df_sites = split_merge_site(df_nonneg.copy())
display(df_sites[['site_id', 'site_type', 'site_num', 'site_label']].drop_duplicates().sort_values('site_id'))

**Intended**: 6 lines, e.g.`SITE_COM_001 → COM → 001 → Commercial #001`.

---
## Part 10 — Calculated columns (formula)

`énergie (kWh) = puissance (MW) × 0,25 h × 1000`(measured every 15 minutes).

Data Wrangler on`df_sites`→ **Create column from formula** (or **Custom operation**):`energy_kwh` = `df['consumption_MW'] * 0.25 * 1000`then`date` = `df['timestamp'].dt.date`.

In [ ]:
def add_calculated_columns(df_sites):
    # Energie (kWh) sur un intervalle de 15 min
    df_sites['energy_kwh'] = df_sites['consumption_MW'] * 0.25 * 1000
    df_sites['date'] = df_sites['timestamp'].dt.date
    return df_sites

df_calc = add_calculated_columns(df_sites.copy())
display(df_calc[['timestamp', 'site_id', 'consumption_MW', 'energy_kwh', 'date']].head())

**Verification**:`0.865` MW → `energy_kwh = 216.25`.

---
## Part 11 — Aggregating with Group by (DataFrame and Separate Files)

Data Wrangler on`df_calc` → **Group by column and aggregate** : group by `site_id`, `date` ; `consumption_MW` → Mean, Max, Count ; `energy_kwh` → Sum ; `voltage_v`→ Mean. Then **Add code to notebook** and rename output to keep`df_calc` intact.

### 11.1 — Daily aggregate

In [ ]:
def aggregate_daily(df_calc):
    df = df_calc.groupby(['site_id', 'date']).agg(
        consumption_MW_mean=('consumption_MW', 'mean'),
        consumption_MW_max=('consumption_MW', 'max'),
        consumption_MW_count=('consumption_MW', 'count'),
        energy_kwh_sum=('energy_kwh', 'sum'),
        voltage_v_mean=('voltage_v', 'mean'),
    ).reset_index()
    return df

df_daily = aggregate_daily(df_calc.copy())
display(df_daily)
print(df_daily.shape)

**Intended**:`(180, 7)` — 6 sites × 30 jours ; `df_calc`and`df_daily` coexistent.

### 11.2 — Backup: Separate files and table

In [ ]:
import os

out_dir = "/lakehouse/default/Files/output"
os.makedirs(out_dir, exist_ok=True)

# Fichiers CSV séparés : détail nettoyé et agrégat journalier
df_calc.to_csv(f"{out_dir}/consumption_clean.csv", index=False)
df_daily.to_csv(f"{out_dir}/consumption_daily_site.csv", index=False)

# Table Delta de l'agrégat
spark.createDataFrame(df_daily).write.mode("overwrite").format("delta").saveAsTable("consumption_daily_site")

**Intended**: **Files →output** contains`consumption_clean.csv`and`consumption_daily_site.csv`(click **Refresh**); Table **`consumption_daily_site`** (180 lines) appears under **Tables**.

---
## Part 12 — Copilot in Data Wrangler

**Prerequisite**: Fabric capacity **F2+** (or P) assigned to workspace (not Trial), parameter holding Copilot activated. Otherwise, go to Part 13 (the manual code below gives the same result).

**Operation 1** — Data Wrangler on`df_calc` → **Custom operation** → zone **Copilot**, prompt :

```text
Add a column named period with the value 'night' when the hour of timestamp is between 22 and 5 (inclusive of 22, exclusive of 6), otherwise 'day'
```
**Run Preview** → reread the code → **Apply**.

**Operation 2** — new **Custom operation**, prompt:

```text
Add a boolean column is_outlier that is True when consumption_MW is greater than the mean plus 3 standard deviations of consumption_MW computed per site_id
```
Check that the calculation is **by site** → **Run Preview** → **Apply** → **Add code to notebook**.

Reference code (handwritten) to compare with Copilot:

In [ ]:
df_copilot = df_calc.copy()

# Opération 1 : période jour/nuit
df_copilot['period'] = df_copilot['timestamp'].dt.hour.apply(lambda h: 'night' if h >= 22 or h < 6 else 'day')

# Opération 2 : valeurs atypiques par site (> moyenne + 3 écarts-types)
g = df_copilot.groupby('site_id')['consumption_MW']
df_copilot['is_outlier'] = df_copilot['consumption_MW'] > (g.transform('mean') + 3 * g.transform('std'))

print(df_copilot['period'].value_counts())
print(df_copilot['is_outlier'].value_counts())

| Question | Reflex |
| --- | --- |
| Does the code do exactly what I asked? | Read before **Apply** |
| Logic by site or global? | Reword the prompt if ambiguous |
| Is the result identical to manual writing? | Comparer (`value_counts`, `.equals()`) |

> Prompts short, one action, explicit column names; the rapids in English usually give better results.

---
## Part 13 — Autonomous Challenge

From`df_calc` :
1. Calculate (Data Wrangler → Group by) consumption **average and maximum by`site_type`per hour**`hour` de `timestamp`)
2. Write the result in`Files/output/consumption_hourly_type.csv`
3. Add:`assert df_hourly['consumption_MW_mean'].notna().all()`

In [ ]:
# Votre code ici

## Summary

| Operation | Effet |
| --- | --- |
| Filter rows | Delete lines`ERROR` |
| Change column type → float | `consumption_mw` exploitable |
| Custom operation (to_datetime) | `timestamp`unified |
| Add code to notebook | Python code reusable |
| Drop duplicate rows | 17 271 → 16 450 lines |
| Find and replace / Custom op | `-999`and negatives →`NaN` |
| Split text + merge | `site_type`, `site_num`, `site_label` |
| Formule | `energy_kwh`, `date` |
| Group by | `df_daily`(180 lines) → CSV + Delta |
| Copilot | `period`,`is_outlier`in natural language |